# ДЗ 3. Поиск похожих птиц: эмбеддинги и HNSW

Нужно найти в базе фотографии птиц того же вида, что на картинке-запросе.
Данные — **CUB-200-2011**, 200 видов птиц. Закрепляем лекцию и семинар 6:
**эмбеддинги → ближайшие соседи → triplet loss → ускорение поиска**.

В этом ДЗ **7 небольших заданий с пропусками**. Загрузчик, модель, обучение,
визуализации и сравнение поиска уже написаны. Новых лоссов здесь нет.

| Оценка | Что выполнить |
|---|---|
| «3» | задания 1–2 и готовый опыт с поиском по замороженным признакам |
| «4» | дополнительно задания 3–5 и готовое обучение с triplet loss |
| «5» | дополнительно задания 6–7 и готовое сравнение полного перебора с HNSW |

**Оценка зависит от корректного кода и выполненных опытов, а не от порога метрики.**
Сдайте этот ноутбук с выводами и картинками. Отдельного отчёта и самостоятельного
кода для экспериментов не требуется. Для полной сдачи выполните Restart & Run All.

Используем один предобученный **ResNet18** и обучаем только линейную голову.
Признаки изображений извлекаются один раз; дальше обучение работает с готовыми
тензорами. В Colab включите GPU для этого первого прохода. Поиск сравнивается
**на CPU**, независимо от того, где извлекались признаки.

Данные: папка `data/` или архив `hw3_data.zip` рядом с ноутбуком (около 190 МБ).
Если их нет, готовая ячейка скачает архив. Для первого запуска также нужны
интернет для весов ResNet18 и установка `hnswlib`.

## Шпаргалка: какие тензоры здесь встречаются

`Q: [B, D]` — B запросов; `X: [N, D]` — N векторов базы; D — длина эмбеддинга.
`y: [B]` — номера видов. Номер строки в базе и номер вида — **разные вещи**.

| Операция | Для чего нужна |
|---|---|
| `F.normalize(Z, dim=1)` | сделать длину каждого вектора равной 1 |
| `Q @ X.T` | сходства всех запросов со всеми объектами: `[B, N]` |
| `S.topk(k, dim=1).indices` | индексы k наибольших элементов каждой строки |
| `y[:, None] == y[None, :]` | матрица совпадения меток `[B, B]` |
| `torch.eye(B, dtype=torch.bool, device=y.device)` | маска диагонали: объект сравнивается с собой |
| `~mask`, `a & b` | НЕ и И для булевых масок |
| `D.masked_fill(~mask, value)` | заменить недопустимые расстояния перед min/max |
| `.max(dim=1).values`, `.min(dim=1).values` | максимум/минимум в каждой строке |
| `.any(dim=1)` | есть ли хотя бы один True в строке |
| `F.relu(t)`, `t.clamp_min(0)` | обнулить отрицательные значения |
| `tensor.detach().cpu().numpy()` | передать данные библиотеке, работающей с NumPy |

Для единичных векторов скалярное произведение — косинусное сходство:
**больше — ближе**. Квадрат евклидова расстояния равен `2 − 2 * similarity`:
**меньше — ближе**. Эти два способа дают одинаковый порядок соседей.

In [ ]:
import copy
import importlib.util
import json
import random
import subprocess
import sys
import time
import urllib.parse
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T

if importlib.util.find_spec("hnswlib") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "hnswlib==0.8.0"], check=True)
import hnswlib


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_all(0)
torch.set_num_threads(4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PASSED, RUNS = {}, {}
print("PyTorch:", torch.__version__, "| извлечение признаков и обучение:", device)

## 1. Полный перебор своими руками

Поиск пока не знает о видах птиц. Он получает **нормированные** векторы и
возвращает номера строк базы, наиболее похожих на запросы.

### Задание 1. Точный top-k

Вычислите `[B, N]` сходств, затем выберите k наибольших в каждой строке.
`exclude_ids` — необязательный тензор `[B]`: какую строку базы нельзя возвращать
для каждого запроса. Если запросы взяты из самой базы, так исключают поиск себя.
Замените соответствующие сходства на `-inf` **до** `topk`.

В основных опытах запросы и база состоят из разных фотографий, поэтому там
`exclude_ids=None`. В тесте отдельно проверяется исключение себя.

In [ ]:
@torch.no_grad()
def exact_search(Q, X, k, exclude_ids=None):
    """Q: float [B,D], X: float [N,D], строки единичной длины, одно устройство.
    k: число соседей, 1 <= k <= N (с исключением: k < N).
    exclude_ids: long [B] или None. Выход: long [B,k], индексы строк X,
    от самого близкого к более далёким. Метки видов сюда не передаются.
    """

    # TODO: матрица сходств, исключение указанных строк, индексы top-k.
    raise NotImplementedError

In [ ]:
_X = F.normalize(torch.tensor([[1., 0.], [0., 1.], [-1., 0.], [.6, .8]]), dim=1)
_Q = F.normalize(torch.tensor([[1., .1], [.1, 1.]]), dim=1)
assert torch.equal(exact_search(_Q, _X, 2), torch.tensor([[0, 3], [1, 3]])),     "Нужны два наиболее похожих объекта базы для каждого запроса"
assert torch.equal(exact_search(_X[:2], _X, 1, torch.tensor([0, 1])), torch.tensor([[3], [3]])),     "Сам запрос должен исключаться до выбора top-k"
assert torch.equal(exact_search(_X[[3, 0]], _X, 1, torch.tensor([3, 0])), torch.tensor([[1], [3]])),     "Исключаем указанный id базы, а не обязательно диагональ"
assert exact_search(_Q[:1], _X, 1).shape == (1, 1), "Не теряйте размерности при B=1 и k=1"
PASSED[1] = True
print("Задание 1: OK")

### Задание 2. Recall@k по видам птиц

Теперь у результатов поиска появится смысл. Для каждого запроса проверяем:
**есть ли среди первых k соседей хотя бы одна птица того же вида?**
Recall@k — доля запросов, для которых ответ «да».

Например, для трёх запросов совпадение нашлось у двух: Recall@k = 2/3.
Это не доля правильных картинок среди всех возвращённых соседей.

In [ ]:
def species_recall(neighbours, query_labels, database_labels, k=1):
    """neighbours: long [B,K] — индексы в базе, K >= k.
    query_labels: long [B], database_labels: long [N] — номера видов.
    Все тензоры на CPU. Выход: одно Python-число от 0 до 1.
    """

    # TODO: метки найденных соседей → совпадения [B,k] → успех по запросам → среднее.
    raise NotImplementedError

In [ ]:
_nn = torch.tensor([[0, 1], [2, 3], [0, 3]])
_qy = torch.tensor([10, 20, 30])
_xy = torch.tensor([10, 30, 30, 20])
assert abs(species_recall(_nn, _qy, _xy, k=1) - 1/3) < 1e-6
assert abs(species_recall(_nn, _qy, _xy, k=2) - 2/3) < 1e-6,     "Успех — хотя бы один правильный сосед; усредняем по запросам"
assert species_recall(torch.tensor([[1]]), torch.tensor([30]), _xy) == 1.
PASSED[2] = True
print("Задание 2: OK")

### Готовый опыт: птицы и замороженные признаки

Учимся на 80 видах из первых 100; ещё 20 видов оставляем для валидации.
Тест — виды 101–200. **Виды между train, val и test не пересекаются.**
Так проверяется перенос поиска на виды, которых не было при обучении головы.

Внутри val и test примерно 20% фотографий каждого вида становятся запросами,
остальные — базой. Одна и та же фотография не может попасть в обе части.
**Виды в запросах и базе совпадают:** у каждого запроса есть правильные ответы.

Код ниже готов. ResNet18 выдаёт 512 признаков для каждого изображения.
Используется обычный average pooling и L2-нормировка. Бэкбон работает в `eval`
и не обучается; извлечённые признаки сохраняются в памяти.

In [ ]:
DATA = Path("data")
DATA_URL = "https://disk.yandex.ru/d/5Dr1V-uDxJuOqw"


def ensure_data():
    if (DATA / "train").is_dir() and (DATA / "test").is_dir():
        return
    archive = Path("hw3_data.zip")
    if not archive.exists():
        api = "https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key="
        with urllib.request.urlopen(api + urllib.parse.quote(DATA_URL, safe=""), timeout=60) as response:
            url = json.load(response)["href"]
        print("Скачиваем hw3_data.zip…")
        urllib.request.urlretrieve(url, archive)
    with zipfile.ZipFile(archive) as z:
        assert all(n.startswith("data/") and ".." not in Path(n).parts for n in z.namelist())
        z.extractall(".")
    assert (DATA / "train").is_dir() and (DATA / "test").is_dir()


def read_files(split):
    files, labels = [], []
    for directory in sorted((DATA / split).iterdir()):
        if directory.is_dir():
            for path in sorted(directory.glob("*.jpg")):
                files.append(path)
                labels.append(int(directory.name[:3]))
    return files, torch.tensor(labels)


ensure_data()
trainval_files, trainval_y = read_files("train")
test_files, test_y = read_files("test")
assert len(trainval_files) == 5864 and len(test_files) == 5924, "Ожидается пакет CUB из ДЗ3"
val_classes = np.random.default_rng(0).choice(np.arange(1, 101), 20, replace=False)
is_val = torch.tensor(np.isin(trainval_y.numpy(), val_classes))
train_files = [f for f, flag in zip(trainval_files, is_val) if not flag]
val_files = [f for f, flag in zip(trainval_files, is_val) if flag]
train_y, val_y = trainval_y[~is_val], trainval_y[is_val]
assert not (set(train_y.tolist()) & set(val_y.tolist()))
assert not (set(trainval_y.tolist()) & set(test_y.tolist()))
for name, files, labels in [("train", train_files, train_y), ("val", val_files, val_y),
                            ("test", test_files, test_y)]:
    print(f"{name}: {len(files)} фото, {len(labels.unique())} видов")

fig, axes = plt.subplots(2, 4, figsize=(12, 5))
for row, files, title in zip(axes, [train_files, test_files], ["Train", "Test"]):
    ids = np.random.default_rng(3).choice(len(files), 4, replace=False)
    for ax, i in zip(row, ids):
        with Image.open(files[i]) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(title + ": " + files[i].parent.name[:3])
        ax.axis("off")
fig.suptitle("Виды из теста не используются при обучении головы")
fig.tight_layout()
plt.show()

In [ ]:
transform = T.Compose([
    T.Resize(180), T.CenterCrop(160), T.ToTensor(),
    T.Normalize([.485, .456, .406], [.229, .224, .225]),
])


class BirdImages(Dataset):
    def __init__(self, files):
        self.files = files
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        with Image.open(self.files[i]) as image:
            return transform(image.convert("RGB"))


backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
backbone.fc = nn.Identity()
backbone = backbone.to(device).eval()


@torch.no_grad()
def extract_features(files):
    loader = DataLoader(BirdImages(files), batch_size=128, shuffle=False, num_workers=0)
    return F.normalize(torch.cat([backbone(batch.to(device)).cpu() for batch in loader]), dim=1)


t0 = time.perf_counter()
F_train = extract_features(train_files)
F_val = extract_features(val_files)
F_test = extract_features(test_files)
print(f"Признаки: {time.perf_counter()-t0:.1f} с; train {tuple(F_train.shape)}")
del backbone
if device.type == "cuda":
    torch.cuda.empty_cache()


def split_queries(labels, seed):
    rng = np.random.default_rng(seed)
    query, database = [], []
    for c in labels.unique():
        ids = rng.permutation(torch.where(labels == c)[0].numpy())
        n_query = max(1, len(ids) // 5)
        query.extend(ids[:n_query]); database.extend(ids[n_query:])
    query, database = torch.tensor(query), torch.tensor(database)
    assert not (set(query.tolist()) & set(database.tolist()))
    return query, database


val_q, val_db = split_queries(val_y, 1)
test_q, test_db = split_queries(test_y, 2)
print(f"Val: {len(val_q)} запросов / {len(val_db)} в базе")
print(f"Test: {len(test_q)} запросов / {len(test_db)} в базе")


def evaluate_features(Z, y, q_ids, db_ids, k=5):
    neighbours = exact_search(Z[q_ids], Z[db_ids], k)
    return species_recall(neighbours, y[q_ids], y[db_ids]), neighbours


def show_neighbours(files, y, q_ids, db_ids, neighbours, title, seed=7):
    # Один seed даёт одни и те же запросы до и после обучения.
    rows = np.random.default_rng(seed).choice(len(q_ids), 4, replace=False)
    fig, axes = plt.subplots(4, 5, figsize=(12, 9))
    for row, r in zip(axes, rows):
        ids = [int(q_ids[r])] + db_ids[neighbours[r, :4]].tolist()
        for j, (ax, i) in enumerate(zip(row, ids)):
            with Image.open(files[i]) as image:
                ax.imshow(image.convert("RGB"))
            color = "black" if j == 0 else ("forestgreen" if y[i] == y[q_ids[r]] else "firebrick")
            ax.set_title(("Запрос" if j == 0 else f"Сосед {j}") + f" · вид {y[i].item()}", color=color)
            ax.axis("off")
    fig.suptitle(title + "\nЗелёный — тот же вид, красный — другой")
    fig.tight_layout()
    plt.show()


baseline_val, baseline_nn = evaluate_features(F_val, val_y, val_q, val_db)
print(f"Замороженный ResNet18: val Recall@1 = {baseline_val:.3f}")
show_neighbours(val_files, val_y, val_q, val_db, baseline_nn, "До обучения: точный поиск на val")
RUNS["baseline"] = True

## 2. Обучаем эмбеддинги знакомым triplet loss

PK-сэмплер определяет **состав батча**: P видов, K фотографий каждого.
Batch-hard определяет, **какую тройку выбрать** для каждого anchor внутри батча.
Сам loss заставляет положительный пример быть ближе отрицательного с запасом margin.

### Задание 3. Маски положительных и отрицательных примеров

Для каждого anchor (строка) отметьте допустимые примеры (столбцы):
- `pos`: тот же вид, **кроме самого anchor**;
- `neg`: другой вид.

Обе маски имеют форму `[B, B]`. На диагонали обеих должен быть False.

In [ ]:
def pair_masks(y):
    """y: long [B] — вид каждой картинки. Выход: две bool-маски [B,B]: pos, neg."""

    # TODO: сравнение меток, маска диагонали, положительные и отрицательные пары.
    raise NotImplementedError

In [ ]:
_pos, _neg = pair_masks(torch.tensor([4, 4, 9]))
assert _pos.dtype == _neg.dtype == torch.bool
assert torch.equal(_pos, torch.tensor([[False, True, False], [True, False, False], [False, False, False]]))
assert torch.equal(_neg, torch.tensor([[False, False, True], [False, False, True], [True, True, False]]))
assert not (pair_masks(torch.tensor([1]))[0] | pair_masks(torch.tensor([1]))[1]).any()
PASSED[3] = True
print("Задание 3: OK")

### Задание 4. Выбор batch-hard тройки

На входе квадратные евклидовы расстояния `D: [B, B]` и маски из задания 3.
Для каждого anchor выберите:
- `d_ap`: **максимальное** расстояние среди положительных;
- `d_an`: **минимальное** расстояние среди отрицательных.

Перед максимумом замените недопустимые элементы на `-inf`, перед минимумом —
на `+inf`. Умножение на нулевую маску не подходит для минимума!

`valid: [B]` отмечает anchor, у которого есть хотя бы один positive **и** negative.
Расстояния невалидных anchor в следующем задании не используются.

In [ ]:
def hardest_distances(D, pos, neg):
    """D: float [B,B], pos/neg: bool [B,B], на одном устройстве.
    Выход: d_ap [B], d_an [B], valid bool [B]. Не отрывайте расстояния от autograd.
    """

    # TODO: максимум по положительным, минимум по отрицательным, маска valid.
    raise NotImplementedError

In [ ]:
_D = torch.tensor([[0., .4, .9, .2], [.4, 0., .6, .8], [.9, .6, 0., .3], [.2, .8, .3, 0.]], requires_grad=True)
_p, _n = pair_masks(torch.tensor([0, 0, 0, 1]))
_ap, _an, _valid = hardest_distances(_D, _p, _n)
assert torch.equal(_valid, torch.tensor([True, True, True, False]))
assert torch.allclose(_ap[_valid], torch.tensor([.9, .6, .9]))
assert torch.allclose(_an[_valid], torch.tensor([.2, .8, .3]))
(_ap[_valid].sum() + _an[_valid].sum()).backward()
assert _D.grad is not None and _D.grad.abs().sum() > 0, "Сохраняйте вычислительный граф"
PASSED[4] = True
print("Задание 4: OK")

### Задание 5. Triplet loss

Для каждой валидной тройки:

$$L_i = \max(0,\ d_{ap,i} - d_{an,i} + m).$$

В этом ДЗ используем **квадрат** евклидова расстояния и margin $m=0.2$.
Loss — среднее **по всем валидным anchor, включая тройки с нулевым loss**.
Пример: потери троек `[0.6, 0.0]` дают среднее `0.3`.

Сборка матрицы расстояний уже дана. Сначала отберите `valid`, затем считайте loss,
чтобы `inf` от отсутствующих соседей не попали в вычисление.
Если валидных троек нет, верните `Z.sum() * 0.0` — ноль, связанный с графом.

In [ ]:
def batch_hard_loss(Z, y, margin=0.2):
    """Z: float [B,D] — нормированные эмбеддинги, y: long [B].
    Выход: скалярный tensor с градиентом, пригодный для backward().
    """
    distances = (2 - 2 * (Z @ Z.T)).clamp_min(0)
    pos, neg = pair_masks(y)
    d_ap, d_an, valid = hardest_distances(distances, pos, neg)

    # TODO: случай без троек; отбор valid; hinge с margin; среднее.
    raise NotImplementedError

In [ ]:
# Независимый маленький эталон с циклами: понятен даже без векторных масок.
def reference_triplet(Z, y, margin):
    terms = []
    for i in range(len(y)):
        positives = [j for j in range(len(y)) if y[j] == y[i] and j != i]
        negatives = [j for j in range(len(y)) if y[j] != y[i]]
        if positives and negatives:
            ap = torch.stack([(Z[i] - Z[j]).square().sum() for j in positives]).max()
            an = torch.stack([(Z[i] - Z[j]).square().sum() for j in negatives]).min()
            terms.append((ap - an + margin).clamp_min(0))
    return torch.stack(terms).mean() if terms else Z.sum() * 0


_z = F.normalize(torch.randn(7, 5, generator=torch.Generator().manual_seed(10)), dim=1).requires_grad_()
_y = torch.tensor([0, 0, 0, 1, 1, 2, 3])
for _margin in (.0, .2, 1.):
    _got = batch_hard_loss(_z, _y, _margin)
    _want = reference_triplet(_z, _y, _margin)
    assert torch.allclose(_got, _want, atol=1e-6), "Проверьте margin, valid и усреднение"
    # Градиент сравниваем по касательным к единичной сфере: Z в реальном обучении нормируется.
    _raw = _z.detach().clone().requires_grad_()
    _norm = F.normalize(_raw, dim=1)
    _g = torch.autograd.grad(batch_hard_loss(_norm, _y, _margin), _raw, retain_graph=True)[0]
    _r = torch.autograd.grad(reference_triplet(_norm, _y, _margin), _raw)[0]
    assert torch.allclose(_g, _r, atol=2e-6), "Нельзя detach()/item() внутри loss"
# Здесь есть и нулевые, и ненулевые потери: среднее нельзя брать только по активным.
_mix = F.normalize(torch.tensor([[1., 0.], [.8, .6], [0., 1.], [-1., 0.]]), dim=1)
assert torch.allclose(batch_hard_loss(_mix, torch.tensor([0, 0, 1, 1])),
                      reference_triplet(_mix, torch.tensor([0, 0, 1, 1]), .2))
for _labels in [torch.arange(7), torch.zeros(7, dtype=torch.long)]:
    _zero = batch_hard_loss(_z, _labels)
    assert _zero.item() == 0 and _zero.requires_grad
    assert torch.equal(torch.autograd.grad(_zero, _z)[0], torch.zeros_like(_z))
PASSED[5] = True
print("Задание 5: OK")

### Готовый опыт: короткое обучение головы

Все пропуски про обучение закончились. Код ниже уже готов:

1. PK-сэмплер собирает **16 видов × 4 фотографии = 64 примера**.
2. Линейная голова преобразует 512 признаков в 512; после неё — L2-нормировка.
3. Обучаем голову одним batch-hard triplet loss. Бэкбон остаётся замороженным.
4. Сохраняем веса с лучшим Recall@1 на val, затем показываем соседей для тех же запросов.

Голова начинает с единичной матрицы: до первого шага она сохраняет исходные признаки.
Исходный вариант тоже участвует в выборе по val. Если обучение его не улучшит,
останется исходная голова — это допустимый результат, а не причина подбирать по test.
Тестовые метки не используются для выбора весов и параметров обучения.

In [ ]:
def pk_batches(labels, P=16, K=4, seed=0):
    rng = np.random.default_rng(seed)
    labels = labels.cpu().numpy()
    classes = np.unique(labels)
    by_class = {c: np.flatnonzero(labels == c) for c in classes}
    for _ in range(len(labels) // (P * K)):
        ids = []
        for c in rng.choice(classes, P, replace=False):
            ids.extend(rng.choice(by_class[c], K, replace=len(by_class[c]) < K))
        yield torch.tensor(ids)


_example = next(pk_batches(train_y))
_classes, _counts = train_y[_example].unique(return_counts=True)
print("Пример PK-батча: виды", _classes.tolist())
print("Количество картинок каждого вида:", _counts.tolist())


class EmbeddingHead(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.linear = nn.Linear(dim, dim, bias=False)
        nn.init.eye_(self.linear.weight)
    def forward(self, features):
        return F.normalize(self.linear(features), dim=1)


@torch.no_grad()
def apply_head(model, features):
    model.eval()
    return torch.cat([model(chunk.to(device)).cpu() for chunk in features.split(512)])


seed_all(1)
head = EmbeddingHead(F_train.shape[1]).to(device)
optimizer = torch.optim.Adam(head.parameters(), lr=1e-4)
train_features_device = F_train.to(device)
train_labels_device = train_y.to(device)
EPOCHS = 15
history = []
best_val, best_epoch = baseline_val, 0
best_state = copy.deepcopy(head.state_dict())
t0 = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    head.train()
    losses = []
    for ids in pk_batches(train_y, seed=epoch):
        ids = ids.to(device)
        loss = batch_hard_loss(head(train_features_device[ids]), train_labels_device[ids])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    val_score, _ = evaluate_features(apply_head(head, F_val), val_y, val_q, val_db)
    history.append((float(np.mean(losses)), val_score))
    if val_score > best_val:
        best_val, best_epoch = val_score, epoch
        best_state = copy.deepcopy(head.state_dict())
    if epoch == 1 or epoch % 5 == 0:
        print(f"Эпоха {epoch:2}: loss {np.mean(losses):.3f}, val Recall@1 {val_score:.3f}")
head.load_state_dict(best_state)
print(f"Обучение: {time.perf_counter()-t0:.1f} с. Выбранная эпоха: {best_epoch}; val {best_val:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(range(1, EPOCHS+1), np.array(history)[:, 0], marker=".")
axes[0].set(xlabel="Эпоха", ylabel="Triplet loss", title="Обучение головы")
axes[1].plot(range(1, EPOCHS+1), np.array(history)[:, 1], marker=".", label="Голова")
axes[1].axhline(baseline_val, color="gray", linestyle="--", label="Исходные признаки")
axes[1].set(xlabel="Эпоха", ylabel="Recall@1", title="Валидация на других видах")
axes[1].legend()
fig.tight_layout(); plt.show()

Z_val = apply_head(head, F_val)
_, trained_nn = evaluate_features(Z_val, val_y, val_q, val_db)
show_neighbours(val_files, val_y, val_q, val_db, trained_nn,
                f"После выбора головы: точный поиск на val (эпоха {best_epoch})")
Z_test = apply_head(head, F_test)
test_before, _ = evaluate_features(F_test, test_y, test_q, test_db)
test_after, test_nn = evaluate_features(Z_test, test_y, test_q, test_db)
print(f"Test Recall@1: исходные признаки {test_before:.3f}; выбранная голова {test_after:.3f}")
show_neighbours(test_files, test_y, test_q, test_db, test_nn, "Выбранная голова: точный поиск на test")
RUNS["training"] = len(history) == EPOCHS and bool(np.isfinite(history).all())

## 3. Готовый HNSW вместо полного перебора

Теперь **эмбеддинги зафиксированы**. Меняем только способ поиска в базе.
Полный перебор сравнивает каждый запрос со всеми N векторами. HNSW заранее
строит граф соседства и при запросе просматривает лишь часть базы.
Сам HNSW писать не нужно — используем библиотеку `hnswlib`.

Памятка по API ([документация](https://github.com/nmslib/hnswlib/tree/v0.8.0)):

| Вызов | Что делает |
|---|---|
| `hnswlib.Index(space="cosine", dim=D)` | создаёт объект индекса для косинусного расстояния |
| `index.init_index(max_elements=N, M=16, ef_construction=100, random_seed=0)` | задаёт вместимость и параметры построения графа |
| `index.add_items(X, ids, num_threads=1)` | добавляет векторы `[N,D]` и их идентификаторы `[N]` |
| `index.set_ef(ef)` | задаёт `efSearch` — размер списка кандидатов при поиске |
| `ids, distances = index.knn_query(Q, k=k, num_threads=1)` | находит k соседей для каждого запроса |

`M` управляет числом связей графа, `ef_construction` — тщательностью построения.
Их оставляем фиксированными. **`efSearch` меняем уже после построения:** больше
кандидатов обычно означает более точный и медленный поиск; `efSearch >= k`.
В `hnswlib` этот параметр называется `ef`, а не `nprobe` (тот относится к IVF).

Библиотека принимает NumPy-массивы. При `space="cosine"` возвращаемые расстояния
равны `1 − cosine_similarity`: меньшие значения соответствуют более близким соседям.

### Задание 6. Построить индекс через API

Объект индекса уже создан. Инициализируйте его и добавьте все векторы базы.
В качестве id используйте **номера строк** `np.arange(len(X))`, а не метки видов.
Параметры: `M=16`, `ef_construction=100`, `random_seed=0`, один поток вставки.

In [ ]:
def build_hnsw(X):
    """X: numpy float32 [N,D], нормированные векторы базы.
    Выход: hnswlib.Index с N объектами, id которых равны 0..N-1.
    """
    index = hnswlib.Index(space="cosine", dim=X.shape[1])

    # TODO: инициализировать индекс; добавить X и id строк; вернуть индекс.
    raise NotImplementedError

In [ ]:
_hx = F.normalize(torch.randn(64, 8, generator=torch.Generator().manual_seed(3)), dim=1).numpy()
_index = build_hnsw(_hx)
assert isinstance(_index, hnswlib.Index) and _index.space == "cosine"
assert _index.get_current_count() == len(_hx)
assert set(_index.get_ids_list()) == set(range(len(_hx))), "id — номера строк базы"
assert np.allclose(_index.get_items([3, 17]), _hx[[3, 17]], atol=1e-6), "В индекс нужно добавить именно X"
PASSED[6] = True
print("Задание 6: OK")

### Задание 7. Выполнить поиск через API

Установите переданный `ef`, вызовите `knn_query` с одним потоком и верните
только индексы соседей `[B,k]`. Граф заново строить не нужно.

**Это последний пропуск в ДЗ.** Сравнение ниже полностью готово: достаточно
запустить его после своих функций.

In [ ]:
def hnsw_search(index, Q, k=10, ef=50):
    """index: готовый индекс. Q: numpy float32 [B,D]. ef >= k.
    Выход: numpy-массив [B,k] с id строк базы, ближайшие идут первыми.
    """

    # TODO: установить ef; выполнить knn_query; вернуть id, а не расстояния.
    raise NotImplementedError

In [ ]:
_hq = _hx[[7, 2, 31]]
_ids = hnsw_search(_index, _hq, k=3, ef=64)
assert isinstance(_ids, np.ndarray) and _ids.shape == (3, 3)
assert np.issubdtype(_ids.dtype, np.integer), "Нужны id, а не расстояния"
assert np.array_equal(_ids[:, 0], [7, 2, 31]), "Запросы здесь специально совпадают с объектами базы"
assert _index.ef == 64, "Не игнорируйте переданный ef"
_expected = exact_search(torch.from_numpy(_hq), torch.from_numpy(_hx), 3).numpy()
assert np.array_equal(_ids, _expected), "На этом маленьком примере ef=64 позволяет найти точный top-3"
assert hnsw_search(_index, _hq[:1], k=1, ef=10).shape == (1, 1)
assert _index.ef == 10
PASSED[7] = True
print("Задание 7: OK")

### Готовое сравнение: скорость и два смысла качества

Используем **те же test-запросы и ту же базу** для обоих методов.
Показываем две метрики, которые нельзя путать:

1. **Совпадение с точным top-10**: средняя доля общих соседей
   $|HNSW_{10}(q) \cap Exact_{10}(q)|/10$. Порядок внутри десятки не важен.
   Это точность приближённого поиска, её можно считать без меток видов.
2. **Recall@1 по видам**: ближайшая картинка содержит птицу правильного вида?
   Это качество решения задачи. Даже точный поиск не гарантирует правильный вид.

Для полного перебора первая метрика равна 1 по определению, вторая обычно ниже 1.
HNSW может случайно улучшить вторую метрику, пропустив точного, но «чужого» соседа:
это не означает, что он точнее воспроизводит ближайших соседей.

**Как замеряется время:** CPU, один поток у обоих методов, одинаковые float32-векторы.
Перенос данных, извлечение эмбеддингов, построение индекса и подсчёт метрик не входят
во время поиска. Построение индекса измеряется отдельно. Есть прогрев, пять повторов,
порядок методов перемешивается; приводится медиана и диапазон измерений.

Запросы идут пакетами по 64. «мс/запрос» — общее время, делённое на число запросов,
**не задержка одиночного онлайн-запроса**. На маленькой базе векторизованный полный
перебор может быть выгоден: ускорение HNSW не обещано заранее.

In [ ]:
# Все подготовительные операции — вне таймера.
X = Z_test[test_db].contiguous().cpu()
Q = Z_test[test_q].contiguous().cpu()
X_np = np.ascontiguousarray(X.numpy(), dtype=np.float32)
Q_np = np.ascontiguousarray(Q.numpy(), dtype=np.float32)
K = 10
BATCH = 64
EF_VALUES = [10, 30, 100, 300]
REPEATS = 5


def search_in_batches(search, queries):
    parts = [search(queries[start:start+BATCH]) for start in range(0, len(queries), BATCH)]
    return torch.cat(parts) if isinstance(parts[0], torch.Tensor) else np.concatenate(parts)


def neighbour_overlap(found, reference):
    # Совпадение множеств id, а не совпадение позиций в упорядоченном списке.
    found = np.asarray(found)
    reference = np.asarray(reference)
    return float((found[:, :, None] == reference[:, None, :]).any(axis=2).mean())


old_threads = torch.get_num_threads()
try:
    torch.set_num_threads(1)
    t0 = time.perf_counter()
    search_index = build_hnsw(X_np)
    build_seconds = time.perf_counter() - t0
    methods = {"Перебор": lambda: search_in_batches(lambda q: exact_search(q, X, K), Q)}
    for ef in EF_VALUES:
        methods[f"HNSW ef={ef}"] = lambda ef=ef: search_in_batches(
            lambda q: hnsw_search(search_index, q, k=K, ef=ef), Q_np)
    timings = {name: [] for name in methods}
    outputs = {}
    for name, search in methods.items():
        outputs[name] = search()   # прогрев
    rng = np.random.default_rng(0)
    for _ in range(REPEATS):
        for name in rng.permutation(list(methods)):
            start = time.perf_counter()
            outputs[name] = methods[name]()
            timings[name].append((time.perf_counter() - start) * 1000 / len(Q))
finally:
    torch.set_num_threads(old_threads)

reference = outputs["Перебор"].numpy()
results = []
print(f"База {len(X)} × {X.shape[1]}, запросов {len(Q)}, k={K}, пакет {BATCH}, CPU: 1 поток")
print(f"Построение HNSW (один раз): {build_seconds:.3f} с")
print(f"{'Метод':<17} {'мс/запрос':>11} {'min..max':>21} {'Совпадение@10':>15} {'Виды R@1':>10}")
for name in methods:
    ids = torch.as_tensor(np.asarray(outputs[name]).astype(np.int64))
    overlap = neighbour_overlap(ids.numpy(), reference)
    recall = species_recall(ids, test_y[test_q], test_y[test_db])
    median = float(np.median(timings[name]))
    low, high = min(timings[name]), max(timings[name])
    results.append(dict(name=name, ms=median, overlap=overlap, recall=recall))
    print(f"{name:<17} {median:11.4f} {low:9.4f}..{high:<9.4f} {overlap:15.3f} {recall:10.3f}")

hnsw_rows = results[1:]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key, ylabel in zip(axes, ["ms", "overlap", "recall"],
                           ["мс / запрос в пакете", "Совпадение с точным top-10", "Recall@1 по видам"]):
    ax.plot(EF_VALUES, [r[key] for r in hnsw_rows], "o-", label="HNSW")
    ax.axhline(results[0][key], color="gray", linestyle="--", label="Полный перебор")
    ax.set(xlabel="efSearch", ylabel=ylabel, xscale="log")
    ax.set_xticks(EF_VALUES, [str(ef) for ef in EF_VALUES])
    if key in ("overlap", "recall"):
        ax.set_ylim(0, 1.03)  # Не превращаем разницу в один запрос в большой скачок графика.
    ax.grid(alpha=.2)
    ax.legend(fontsize=9)
fig.suptitle("Меняем поиск, эмбеддинги остаются теми же")
fig.tight_layout(); plt.show()
RUNS["benchmark"] = all(np.isfinite([r["ms"], r["overlap"], r["recall"]]).all() for r in results)

### Как прочитать результат

- Найдите изменение времени и совпадения top-10 при росте `efSearch`. На конечной
  выборке цифры могут немного колебаться; не каждый шаг обязан улучшать метрику.
- Сравните два столбца качества: высокая точность поиска не исправляет плохие эмбеддинги.
- Построение HNSW окупается только при достаточном числе запросов. Если база мала
  или запросов мало, полный перебор может быть удобнее.
- Время зависит от CPU, размера базы, размерности вектора и размера пакета.
  Полученные числа описывают этот опыт; они не являются универсальным рейтингом методов.

**Письменные ответы не нужны:** таблица и графики уже входят в сдаваемый ноутбук.

## Итог

Последний студенческий код — `hnsw_search`. Все эксперименты после пропусков готовы.
Для полной сдачи нужны семь пройденных проверок и три выполненных опыта:
поиск до обучения, обучение головы, сравнение точного поиска с HNSW.

In [ ]:
print(f"Проверки: {sum(PASSED.get(i, False) for i in range(1, 8))}/7")
for grade, count, runs in [(3, 2, ["baseline"]),
                            (4, 5, ["baseline", "training"]),
                            (5, 7, ["baseline", "training", "benchmark"])]:
    complete = all(PASSED.get(i, False) for i in range(1, count+1)) and all(RUNS.get(r, False) for r in runs)
    print(f"Уровень «{grade}»: {'выполнен' if complete else 'не завершён'}")